# AgriCL-Net, step 3: disease classifier (Swin-T)

Fine-tunes **Swin-T** (pretrained on ImageNet-22k) on the **112 seen PlantSeg diseases**, using a square crop
around the lesions so the model looks at the symptoms, not the background. Rare diseases are sampled more
often (square-root class balancing).

After training it fits two things on the validation set:
* a **temperature** that makes confidence scores honest (calibration);
* an **unknown threshold** that keeps 95% of known-disease images; anything less confident is answered
  "Unknown, consult an expert".

It then reports top-1/top-3 accuracy and macro-F1 on the seen test split, and how many images of the
**three unseen diseases** are correctly flagged as unknown. This model is also the "joint training" upper
bound for the continual-learning step.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run once with `SMOKE_TEST = True`
(40 images, 1 epoch), then set it to `False` and choose Runtime > Run all. Training resumes after a disconnect.

## Settings

In [ ]:
MODEL = 'swin_tiny_patch4_window7_224.ms_in22k_ft_in1k'   # timm name; fixed 224 px input
BATCH = 32          # lower to 16 if out of memory
EPOCHS = 30
LR = 1e-4
SMOKE_TEST = True   # True: 40 images, 1 epoch. Set False for the real run.

## GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Storage and project code from GitHub

Works on **Colab** and **Kaggle**:
* Colab: everything is kept in Google Drive, `MyDrive/Colab Notebooks/agricl-net`.
* Kaggle: turn on *Settings > Accelerator > GPU T4* and *Settings > Internet*. Results go to
  `/kaggle/working/agricl-net` and are kept when you *Save Version*. Downloads go to `/tmp` (fetched again each
  session). To use an earlier notebook's results (or resume a run), attach that notebook's output with
  *Add Input > Notebook Output*: it is copied in automatically below.

In [ ]:
import os, sys, glob, shutil
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK = '/tmp'                               # scratch space: code, unzipped data
    DRIVE = '/kaggle/working/agricl-net'        # results, saved with each notebook version
    CACHE = '/tmp/agricl-cache'                 # dataset zip and SAM2 checkpoint
    for src in glob.glob('/kaggle/input/**/agricl-net/runs/*', recursive=True):
        dst = f'{DRIVE}/runs/{os.path.basename(src)}'
        if os.path.isdir(src) and not os.path.exists(dst):
            shutil.copytree(src, dst)
            print('copied earlier results:', os.path.basename(src))
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    DRIVE = '/content/drive/MyDrive/Colab Notebooks/agricl-net'   # dataset zip, SAM2 checkpoint and all results
    CACHE = f'{DRIVE}/cache'
os.makedirs(CACHE, exist_ok=True)
os.makedirs(f'{DRIVE}/runs', exist_ok=True)

CODE = f'{WORK}/AGRICL-NET'
if os.path.exists(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q https://github.com/Dev02005/AGRICL-NET.git {CODE}
sys.path.insert(0, CODE)
!git -C {CODE} log -1 --format='code version: %h %s'

## PlantSeg v7 dataset (1.1 GB; on Colab downloaded to Drive once)

In [ ]:
import hashlib
ZIP = f'{CACHE}/plantseg.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O "{ZIP}" "https://zenodo.org/records/17719108/files/plantseg.zip?download=1"

h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        h.update(chunk)
assert h.hexdigest() == '9358a66dff88cdd15c4fe009763c40a3', f'bad download, md5 {h.hexdigest()}: delete {ZIP} and rerun'

DATA = f'{WORK}/data/plantseg'
if not os.path.exists(f'{DATA}/Metadata.csv'):
    !unzip -q "{ZIP}" -d "{WORK}/data"

In [ ]:
!pip -q install timm
RUN = 'classifier_' + MODEL.split('.')[0] + ('_smoke' if SMOKE_TEST else '')
OUT = f'{DRIVE}/runs/{RUN}'
print('results ->', OUT)

## Model check: one batch forward and backward (GPU memory)

In [ ]:
import torch
from agricl.models.classifier import build_classifier

m = build_classifier(112, MODEL).cuda().train()
x = torch.randn(BATCH, 3, 224, 224, device='cuda')
with torch.autocast('cuda', dtype=torch.float16):
    loss = m(x).float().logsumexp(1).mean()
loss.backward()
print(f'parameters: {sum(p.numel() for p in m.parameters()) / 1e6:.1f} M, '
      f'peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB')
del m, x, loss
torch.cuda.empty_cache()

## Train (resumes automatically from Drive)

In [ ]:
from agricl.train_classifier import ClsConfig, train, final_report

cfg = ClsConfig(
    data_root=DATA, out_dir=OUT, model_name=MODEL, batch_size=BATCH, lr=LR,
    epochs=1 if SMOKE_TEST else EPOCHS, limit=40 if SMOKE_TEST else 0, workers=2,
)
model = train(cfg)

## Final evaluation

* `test_seen`: accuracy on known diseases, calibration error before/after temperature scaling, and how many
  known images are wrongly flagged as unknown (about 5% by design).
* `unseen`: share of the three held-out diseases flagged as unknown (higher is better), and AUROC for
  separating known from unseen images by confidence (1.0 = perfect, 0.5 = chance).

In [ ]:
results = final_report(cfg, model)

## Training curve, weakest diseases and confidence on known vs. unseen images

In [ ]:
import json, csv, numpy as np, matplotlib.pyplot as plt
from agricl.calibration import softmax

hist = json.load(open(f'{OUT}/history.json'))
ep = [h['epoch'] for h in hist]
fig, ax = plt.subplots(1, 3, figsize=(15, 3))
ax[0].plot(ep, [h['train_loss'] for h in hist]); ax[0].set_title('train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(ep, [h['val_top1'] for h in hist], label='top-1')
ax[1].plot(ep, [h['val_top3'] for h in hist], label='top-3'); ax[1].legend(); ax[1].set_title('validation accuracy')

cal = json.load(open(f'{OUT}/calibration.json'))
lg = np.load(f'{OUT}/logits.npz')
ax[2].hist(softmax(lg['test'], cal['temperature']).max(1), bins=30, alpha=0.6, density=True, label='known (test)')
ax[2].hist(softmax(lg['unseen'], cal['temperature']).max(1), bins=30, alpha=0.6, density=True, label='unseen diseases')
ax[2].axvline(cal['unknown_threshold'], color='k', ls='--', label='unknown threshold')
ax[2].set_title('calibrated confidence'); ax[2].legend()
plt.show()

rows = sorted(csv.DictReader(open(f'{OUT}/per_class.csv')), key=lambda r: float(r['accuracy']))
print('weakest 10 diseases on the test split:')
for r in rows[:10]:
    print(f"  {float(r['accuracy']):.2f}  ({r['test_images']} images)  {r['disease']}")

## Next
* Smoke test passed? Set `SMOKE_TEST = False` in the settings cell and run all cells.
* Results: `MyDrive/Colab Notebooks/agricl-net/runs/<run name>/` (`best.pt`, `classes.json`, `calibration.json`, `results.json`,
  `per_class.csv`, `logits.npz`).
* The continual-learning step will compare against this model's accuracy as the upper bound.